In [10]:
pip install pandas numpy skyfield

In [11]:
# process_data.py (Final Corrected Version - Reads TLE.txt)

import pandas as pd
import numpy as np
import os
from skyfield.api import load, EarthSatellite
from datetime import timedelta

print("🚀 Starting Step 1: Generating Time-Series Orbit Data from TLE file...")

# --- Part A: Load and Parse the TLE Text File ---
# CORRECTION: Pointing to the correct file with TLE data
DATA_FILE_PATH = '/kaggle/input/starlink-satellite-tlecsv-dataset-april-2025/starlink_tle.txt'

if not os.path.exists(DATA_FILE_PATH):
    print(f"Error: The file was not found at: {DATA_FILE_PATH}")
else:
    # Read the text file line by line
    with open(DATA_FILE_PATH, 'r') as f:
        lines = f.readlines()

    # Parse the TLE data (which comes in 3-line sets)
    tle_data = []
    for i in range(0, len(lines), 3):
        tle_data.append({
            'Satellite_Name': lines[i].strip(),
            'TLE_LINE1': lines[i+1].strip(),
            'TLE_LINE2': lines[i+2].strip()
        })

    master_df = pd.DataFrame(tle_data)
    print(f"✅ Successfully parsed {len(master_df)} satellites from the TLE.txt file.")

    # --- Part B: Generate Positional Data using Skyfield ---
    ts = load.timescale()
    position_data = []

    # We will generate data for the first 15 satellites
    satellites_to_process = master_df.head(15)
    print(f"\n🛰️  Processing and generating orbits for {len(satellites_to_process)} satellites...")

    for index, tle_row in satellites_to_process.iterrows():
        # Create a satellite object from the TLE lines
        satellite = EarthSatellite(tle_row['TLE_LINE1'], tle_row['TLE_LINE2'], tle_row['Satellite_Name'], ts)

        # Get the epoch (timestamp) directly from the satellite TLE data
        epoch_time = satellite.epoch.utc_datetime()

        # Generate positions for a 24-hour period, with one point every 10 minutes
        time_range = pd.to_datetime([epoch_time + timedelta(minutes=10 * i) for i in range(144)])

        skyfield_times = ts.from_datetimes(time_range)
        positions = satellite.at(skyfield_times).position.km

        # Store the calculated positions
        for i, t in enumerate(time_range):
            position_data.append({
                'timestamp': t,
                'norad_id': tle_row['Satellite_Name'],
                'x_km': positions[0, i],
                'y_km': positions[1, i],
                'z_km': positions[2, i],
            })

    # --- Part C: Save the Processed Data ---
    processed_df = pd.DataFrame(position_data)
    output_path = '/kaggle/working/processed_orbits.csv'
    processed_df.to_csv(output_path, index=False)

    print(f"\n✅ Data generation complete! A new file with {len(processed_df)} rows has been saved to '{output_path}'.")
    print("\nHere's a preview of your new, rich dataset:")
    print(processed_df.head())

🚀 Starting Step 1: Generating Time-Series Orbit Data from TLE file...
Error: The file was not found at: /kaggle/input/starlink-satellite-tlecsv-dataset-april-2025/starlink_tle.txt


In [12]:
df = pd.read_csv(DATA_FILE_PATH)
print("The actual column names are:")
print(df.columns) # Add this line to see all column names

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/starlink-satellite-tlecsv-dataset-april-2025/starlink_tle.txt'

In [ ]:
pip install plotly

In [ ]:
# visualization.py (Kaggle Version)

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

# This line is crucial for displaying the plot in the notebook
pio.renderers.default = 'notebook'

print("📊 Starting Step 2: 3D Orbit Visualization...")

# --- Part A: Load the Processed Data ---
# Load the clean data file we created in Step 1
try:
    df = pd.read_csv('/kaggle/working/processed_orbits.csv')
    print("✅ Successfully loaded 'processed_orbits.csv'.")
except FileNotFoundError:
    print("❌ Error: 'processed_orbits.csv' not found. Please ensure Step 1 ran successfully.")
    # Stop execution if the file doesn't exist
    exit()

# --- Part B: Create the 3D Plot ---
fig = go.Figure()

# Create a sphere for the Earth using mathematical calculations
u, v = np.mgrid[0:2*np.pi:40j, 0:np.pi:20j]
earth_x = 6371 * np.cos(u) * np.sin(v)
earth_y = 6371 * np.sin(u) * np.sin(v)
earth_z = 6371 * np.cos(v)

# Add the Earth sphere to the plot
fig.add_trace(go.Surface(
    x=earth_x, y=earth_y, z=earth_z,
    colorscale=[[0, 'blue'], [1, 'blue']], # A single solid color for the sphere
    showscale=False,
    name='Earth'
))

# --- Part C: Add Orbits for Each Satellite ---
# Loop through each unique satellite in our data and plot its path
for sat_id in df['norad_id'].unique():
    sat_data = df[df['norad_id'] == sat_id]
    fig.add_trace(go.Scatter3d(
        x=sat_data['x_km'], y=sat_data['y_km'], z=sat_data['z_km'],
        mode='lines',
        name=str(sat_id), # Use the satellite name/ID in the legend
        line=dict(width=3)
    ))

# --- Part D: Configure and Display the Plot ---
# Update the layout for a professional look
fig.update_layout(
    title='Interactive 3D Visualization of Starlink Satellite Orbits',
    scene=dict(
        xaxis_title='X (km)',
        yaxis_title='Y (km)',
        zaxis_title='Z (km)',
        aspectmode='data' # This ensures the Earth is a sphere, not a stretched oval
    ),
    margin=dict(r=10, l=10, b=10, t=40)
)

# Show the plot in your notebook's output
fig.show()

print("\n✅ Visualization complete! You can now interact with the 3D plot above.")

In [ ]:
pip install tensorflow xgboost scikit-learn matplotlib seaborn

In [ ]:
# train_models_final_robust.py (Kaggle Version)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input
import xgboost as xgb

print("🧠 Starting Step 3 (Final Robust Version): Model Training, Validation, and Evaluation...")

# --- Part A: Load the Processed Data ---
try:
    df = pd.read_csv('/kaggle/working/processed_orbits.csv')
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    print("✅ Successfully loaded 'processed_orbits.csv'.")
except FileNotFoundError:
    print("❌ Error: 'processed_orbits.csv' not found. Please ensure Step 1 ran successfully.")
    exit()

# --- Part B: Train LSTM for Trajectory Prediction with Scaling and Evaluation 📈 ---
print("\n--- LSTM Model ---")

# Prepare data for a single satellite
sat_id_for_training = df['norad_id'].unique()[0]
sat_df = df[df['norad_id'] == sat_id_for_training][['x_km', 'y_km', 'z_km']]

# Scale the data for better LSTM performance
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(sat_df)

def create_sequences(data, seq_length):
    xs, ys = [], []
    for i in range(len(data) - seq_length):
        x = data[i:(i + seq_length)]
        y = data[i + seq_length]
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)

SEQ_LENGTH = 10
X, y = create_sequences(scaled_data, SEQ_LENGTH)

# Split data: 70% Train, 15% Validation, 15% Test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)
print(f"LSTM Data Split: Train={len(X_train)}, Validation={len(X_val)}, Test={len(X_test)}")

# Build and train the LSTM model
lstm_model = Sequential([
    Input(shape=(SEQ_LENGTH, 3)),
    LSTM(50, activation='relu'),
    Dense(3)
])
lstm_model.compile(optimizer='adam', loss='mse')
history = lstm_model.fit(X_train, y_train, epochs=50, validation_data=(X_val, y_val), verbose=0, batch_size=16)

# Evaluate on the test set
test_loss = lstm_model.evaluate(X_test, y_test, verbose=0)
print(f"\nFinal Test MSE (on scaled data): {test_loss:.6f}") # Loss will be very small

# Plotting Training vs. Validation Loss
plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('LSTM: Training vs. Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.grid(True)
plt.show()

# Save the final trained model
lstm_model.save('/kaggle/working/lstm_model.h5')
print("✅ LSTM model trained and saved as 'lstm_model.h5'")

# --- Part C: Train XGBoost for Collision Risk with Stratified Split 💥 ---
print("\n\n--- XGBoost Model ---")

sat1_df = df[df['norad_id'] == df['norad_id'].unique()[0]].reset_index()
sat2_df = df[df['norad_id'] == df['norad_id'].unique()[1]].reset_index()
min_len = min(len(sat1_df), len(sat2_df))
risk_df = pd.DataFrame({ 'x1': sat1_df['x_km'][:min_len], 'y1': sat1_df['y_km'][:min_len], 'z1': sat1_df['z_km'][:min_len], 'x2': sat2_df['x_km'][:min_len], 'y2': sat2_df['y_km'][:min_len], 'z2': sat2_df['z_km'][:min_len] })
risk_df['distance'] = np.sqrt((risk_df['x1'] - risk_df['x2'])**2 + (risk_df['y1'] - risk_df['y2'])**2 + (risk_df['z1'] - risk_df['z2'])**2)
risk_df['risk'] = 0 # Start with all events as low risk

# --- FIX IS HERE: Create a few artificial "High Risk" events ---
# Find the indices of the 5 closest approaches
closest_indices = risk_df['distance'].nsmallest(5).index
risk_df.loc[closest_indices, 'risk'] = 1 # Set these as high risk
print(f"Created {len(closest_indices)} artificial 'High Risk' events for training.")
# --- END OF FIX ---

# Use stratify to ensure both classes are in the training set
X_risk = risk_df[['distance']]
y_risk = risk_df['risk']
X_train_xgb, X_test_xgb, y_train_xgb, y_test_xgb = train_test_split(X_risk, y_risk, test_size=0.2, random_state=42, stratify=y_risk)
print(f"XGBoost Data Split: Train={len(X_train_xgb)}, Test={len(X_test_xgb)}")

# Train the XGBoost model
xgb_model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_model.fit(X_train_xgb, y_train_xgb)

# Make predictions and evaluate
y_pred_xgb = xgb_model.predict(X_test_xgb)
accuracy = accuracy_score(y_test_xgb, y_pred_xgb)
print(f"\nModel Accuracy on Test Set: {accuracy * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test_xgb, y_pred_xgb, zero_division=0))

# Plotting the Confusion Matrix
cm = confusion_matrix(y_test_xgb, y_pred_xgb)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Low Risk', 'High Risk'], yticklabels=['Low Risk', 'High Risk'])
plt.title('XGBoost: Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

# Save the final trained model
xgb_model.save_model('/kaggle/working/xgboost_model.json')
print("✅ XGBoost model trained and saved as 'xgboost_model.json'")

In [ ]:
pip install streamlit

In [ ]:
%%writefile app.py
# This command writes the content of this cell to a file named 'app.py'

import streamlit as st
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from tensorflow.keras.models import load_model
import xgboost as xgb
from sklearn.preprocessing import MinMaxScaler
import joblib

# --- Page Configuration ---
st.set_page_config(
    page_title="Space Debris Simulation Dashboard",
    layout="wide",
    initial_sidebar_state="expanded"
)

# --- Load Data and Models ---
@st.cache_data
def load_data():
    df = pd.read_csv('processed_orbits.csv')
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    return df

@st.cache_resource
def load_ml_models():
    # --- FIX IS HERE ---
    # We add compile=False because we only need the model for prediction, not training.
    lstm = load_model('lstm_model.h5', compile=False)
    # --- END OF FIX ---

    xgb_model = xgb.XGBClassifier()
    xgb_model.load_model('xgboost_model.json')
    return lstm, xgb_model

df = load_data()
lstm_model, xgb_model = load_ml_models()
unique_satellites = df['norad_id'].unique()

# --- Sidebar ---
st.sidebar.title("🛰️ Simulation Controls")
selected_satellites = st.sidebar.multiselect(
    "Select Satellites to Display",
    options=unique_satellites,
    default=list(unique_satellites[:3])
)

# --- Main Dashboard ---
st.title("🚀 Space Debris Analysis and Prediction Dashboard")
st.markdown("An interactive platform to model, visualize, and predict satellite movements.")

# --- 3D Orbit Visualization ---
st.header("3D Orbit Visualization")
plot_df = df[df['norad_id'].isin(selected_satellites)]

fig = go.Figure()

# Plot Earth
u, v = np.mgrid[0:2*np.pi:40j, 0:np.pi:20j]
earth_x = 6371 * np.cos(u) * np.sin(v)
earth_y = 6371 * np.sin(u) * np.sin(v)
earth_z = 6371 * np.cos(v)
fig.add_trace(go.Surface(x=earth_x, y=earth_y, z=earth_z, colorscale=[[0, 'blue'], [1, 'blue']], showscale=False))

# Plot satellite orbits
for sat_id in selected_satellites:
    sat_data = plot_df[plot_df['norad_id'] == sat_id]
    fig.add_trace(go.Scatter3d(
        x=sat_data['x_km'], y=sat_data['y_km'], z=sat_data['z_km'],
        mode='lines',
        name=str(sat_id),
        line=dict(width=3)
    ))

fig.update_layout(
    title='Satellite Orbits Around Earth',
    scene=dict(xaxis_title='X (km)', yaxis_title='Y (km)', zaxis_title='Z (km)', aspectmode='data'),
    margin=dict(r=10, l=10, b=10, t=40)
)
st.plotly_chart(fig, use_container_width=True)


# --- ML Predictions Section ---
st.header("🤖 Machine Learning Predictions")
col1, col2 = st.columns(2)

# --- Trajectory Prediction (LSTM) ---
with col1:
    st.subheader("Trajectory Forecasting")
    pred_sat_id = st.selectbox("Select satellite for prediction", options=unique_satellites, index=0)

    if st.button("Predict Next Position"):
        # We need to scale the input data just like we did for training
        scaler = MinMaxScaler()
        sat_data_full = df[df['norad_id'] == pred_sat_id][['x_km', 'y_km', 'z_km']]
        scaler.fit(sat_data_full) # Fit the scaler on all data for this satellite
        scaled_data = scaler.transform(sat_data_full)

        SEQ_LENGTH = 10
        last_sequence = scaled_data[-SEQ_LENGTH:]
        input_sequence = np.expand_dims(last_sequence, axis=0)

        # Make prediction
        prediction_scaled = lstm_model.predict(input_sequence)
        prediction = scaler.inverse_transform(prediction_scaled)

        st.success("Prediction Complete!")
        pred_df = pd.DataFrame(prediction, columns=['Predicted X (km)', 'Predicted Y (km)', 'Predicted Z (km)'])
        st.dataframe(pred_df)

# --- Collision Risk Assessment (XGBoost) ---
with col2:
    st.subheader("Collision Risk Assessment")
    risk_sat_1 = st.selectbox("Select Satellite 1", options=unique_satellites, index=0)
    risk_sat_2 = st.selectbox("Select Satellite 2", options=unique_satellites, index=1)

    if st.button("Assess Collision Risk"):
        if risk_sat_1 == risk_sat_2:
            st.error("Please select two different satellites.")
        else:
            sat1_pos = df[df['norad_id'] == risk_sat_1].iloc[-1][['x_km', 'y_km', 'z_km']].values
            sat2_pos = df[df['norad_id'] == risk_sat_2].iloc[-1][['x_km', 'y_km', 'z_km']].values

            distance = np.linalg.norm(sat1_pos - sat2_pos)
            feature_vector = np.array([[distance]])

            # Predict probability
            risk_prob = xgb_model.predict_proba(feature_vector)[0][1]

            st.metric("Current Distance (km)", f"{distance:,.2f}")
            st.metric("Predicted Collision Risk Probability", f"{risk_prob:.2%}")

            if risk_prob > 0.5:
                st.warning("High risk of close approach detected!")
            else:
                st.success("Low risk of close approach.")

In [ ]:
!npm install -g localtunnel

In [ ]:
!pip install --force-reinstall "streamlit==1.32.2" "plotly==5.15.0"

In [ ]:
!streamlit run app.py & npx localtunnel --port 8501

In [ ]:
!curl https://loca.lt/mytunnelpassword